In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from solver import *

In [ ]:
# Define the symbols and the unknown function
t, x, y, xi, eta = symbols('t x y xi eta', real=True)
u_func = Function('u') 
u = u_func(t, x, y)

# ------------------------------------------------------------------
# Candidate 2D symbols -- uncomment to experiment
# ------------------------------------------------------------------
# symbol_expr = xi**2 + eta**2                                      # free -Laplacian (straight rays)
# symbol_expr = y * xi - x * eta                                    # vortex
# symbol_expr = y * xi - x * eta - 1/2*(xi**2 + eta**2)             # vortex - Laplacian/2
# symbol_expr = (xi**2 + eta**2) * (1 + cos(x)*cos(y))        # variable-coefficient Laplacian
# symbol_expr = (xi*cos(x) + eta*sin(y))                      # anisotropic transport
# symbol_expr = sqrt(xi**2 + eta**2 + 1)                         # relativistic / order-1 elliptic
# symbol_expr = exp(I*(x*xi + y*eta))*(xi**2 + eta**2)        # NUFFT-representable joint term
# symbol_expr = 1/(1 + (1 - xi)**2 + (1 - eta)**2)                  # AAA-representable joint term
# symbol_expr = xi**2 + eta**2 + cos(x)*cos(y)                # -Laplacian + periodic potential

# Examples from Laplace-Beltrami operator - lbo_solver.ipynb
# symbol_expr = 0.4*eta**2 + 0.1*eta*x + 0.1*eta*xi + 0.7*xi**2 - 0.1*xi*y
# symbol_expr = eta**2/(cos(x) + 2)**2 + xi**2
# symbol_expr = (eta**2 + xi**2*(x**2 + y**2 + 1))/(x**2 + y**2 + 1)
# symbol_expr = (eta**2 + xi**2)/cosh(y)**2
# symbol_expr = (xi**2 + eta**2)/2 + atan2(y, x)
# symbol_expr = (xi**2 + eta**2)/2 - (x**2 + y**2) + (x**2 + y**2)**2

# Examples from Hamiltonians
A = 1
B = 1
m = 1
k = 0.5
alpha = 1
beta = 1

eps = 0.01
symbol_expr = ((xi - B*y/2)**2 + (eta + B*x/2)**2)/(2*m) + 0.5*k*(x**2 + y**2)

# symbol_expr = (xi**2 + eta**2)/(2*m) - k/sqrt(x**2 + y**2 + eps)
# symbol_expr = (xi**2 + eta**2)/(2*m) + (A/(x**2 + y**2 + eps))**2
# symbol_expr = sqrt(xi**2 + eta**2 + m**2) - k/sqrt(x**2 + y**2 + eps)
# symbol_expr = (xi**2 + eta**2)/2 - 1/sqrt((x+alpha)**2 + y**2 + eps) - alpha/sqrt((x-1)**2 + y**2 + eps)
# symbol_expr = (xi**2 + eta**2)/2 + alpha*x**3 + beta*x*y

# symbol_expr = (xi**2 + eta**2)/2 + alpha*cos(2*pi*x/3) + beta*cos(2*pi*y/3)

# symbol_expr = (xi**2 + eta**2)/(2*m) + alpha/(x**2 + y**2 + eps) + beta*(x + y)

eq = Eq(I * diff(u, t), psiOp(symbol_expr, u))

# Create the solver
solver = PDESolver(eq)

# Domain
Lx = Ly = 20
Nx = Ny = 256
Lt = 10.0
Nt = 2000

# Initial condition: Gaussian wave packet modulated e^{-x^2 - y^2} e^{i(x + y)}
def f_initial(X, Y):
    return (np.exp(-0.5 * ((X - 1.5)**2 + (Y + 1.0)**2))
            * np.exp(1j * (3.0*X - 2.0*Y))
           )

solver.setup(
    Lx=Lx, Ly=Ly, Nx=Nx, Ny=Ny, Lt=Lt, Nt=Nt,
    initial_condition=f_initial
)

# Solve the equation
frames = solver.solve()


In [ ]:
# Raise the animation size limit to allow large frame counts at high resolution
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None, # 'contour', 'front' or None
    mode='surface', # 'imshow' or 'surface'
    physical=True # True or False
)

HTML(ani.to_jshtml())